In [1]:
## Load dataset

import pandas as pd
import zipfile

path = "TCGA_Reports.csv.zip"

with zipfile.ZipFile(path) as archive:
    csv_name = next(
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    )

    with archive.open(csv_name) as file:
        df = pd.read_csv(file, low_memory=False)

print(df.shape)
print(df.columns.tolist())
print(df.head(3).to_string())

(9523, 2)
['patient_filename', 'text']
                                    patient_filename                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [2]:

import re
import json
import pandas as pd

from typing import Optional
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


# ============================================================
# MODULE 1: TEXT PREPROCESSING
# ============================================================

def preprocess_clinical_text(text: Optional[str]) -> str:
    """
    Normalize clinical text while preserving clinically
    important words, punctuation, and negation.
    """
    if text is None or pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def split_sentences(text: str) -> list:
    """Simple sentence splitter returning character offsets."""
    results = []

    pattern = re.compile(r"[^.!?\n]+(?:[.!?]+|(?=\n)|$)")

    for match in pattern.finditer(text):
        raw = match.group()
        sentence = raw.strip()

        if sentence:
            start = match.start() + len(raw) - len(raw.lstrip())
            end = start + len(sentence)
            results.append({
                "text": sentence,
                "start": start,
                "end": end,
            })

    return results


# ============================================================
# MODULE 2: CLINICAL NAMED ENTITY RECOGNITION
# ============================================================

CANCER_TERMS = {
    "CANCER": [
        "cancer", "malignancy", "malignant neoplasm",
        "carcinoma", "adenocarcinoma", "sarcoma",
        "lymphoma", "leukemia", "leukaemia",
        "melanoma", "myeloma", "metastasis",
        "metastases", "neoplasm",
    ],
    "HISTOLOGY": [
        "adenocarcinoma", "squamous cell carcinoma",
        "small cell carcinoma", "ductal carcinoma",
        "lobular carcinoma", "urothelial carcinoma",
        "papillary carcinoma", "melanoma",
        "lymphoma", "sarcoma",
    ],
    "ANATOMICAL_SITE": [
        "lung", "breast", "colon", "colorectal",
        "rectum", "prostate", "pancreas", "liver",
        "stomach", "kidney", "renal", "bladder",
        "thyroid", "ovary", "ovarian", "uterus",
        "endometrium", "brain", "skin", "bone",
    ],
    "STAGE": [
        "stage i", "stage ii", "stage iii", "stage iv",
        "stage 1", "stage 2", "stage 3", "stage 4",
    ],
    "TREATMENT": [
        "chemotherapy", "radiotherapy", "radiation therapy",
        "immunotherapy", "mastectomy", "tumor resection",
        "tumour resection", "surgical resection",
    ],
}


def clinical_ner(text: str) -> list:
    """
    Dictionary-based clinical NER.

    Returns entity text, entity type, and character offsets.
    Overlapping entity categories are permitted.
    """
    entities = []

    for entity_type, terms in CANCER_TERMS.items():
        for term in terms:
            pattern = (
                r"(?<!\w)" + re.escape(term) + r"(?!\w)"
            )

            for match in re.finditer(
                pattern, text, flags=re.IGNORECASE
            ):
                entities.append({
                    "text": match.group(),
                    "type": entity_type,
                    "start": match.start(),
                    "end": match.end(),
                })

    entities.sort(key=lambda x: (x["start"], x["end"]))
    return entities


# ============================================================
# MODULE 3: NEGATION, UNCERTAINTY AND TEMPORALITY
# ============================================================

NEGATION_CUES = [
    "no evidence of",
    "negative for",
    "without evidence of",
    "no sign of",
    "no signs of",
    "absence of",
    "not identified",
    "not seen",
    "no malignancy",
    "no carcinoma",
]

UNCERTAINTY_CUES = [
    "possible",
    "possibly",
    "probable",
    "suspected",
    "suspicious for",
    "cannot exclude",
    "cannot rule out",
    "may represent",
    "might represent",
    "concerning for",
]

HISTORICAL_CUES = [
    "history of",
    "previously diagnosed",
    "previously treated",
    "prior history",
    "in remission",
    "status post",
]

FAMILY_HISTORY_CUES = [
    "family history of",
    "mother had",
    "father had",
    "sister had",
    "brother had",
]


def detect_clinical_context(sentence: str) -> dict:
    """
    Baseline sentence-level context detection.

    This is not a complete clinical negation algorithm.
    It requires validation against manually annotated reports.
    """
    normalized = sentence.lower()
    normalized = re.sub(r"\s+", " ", normalized).strip()

    family_history = any(
        cue in normalized for cue in FAMILY_HISTORY_CUES
    )

    uncertain = any(
        cue in normalized for cue in UNCERTAINTY_CUES
    )

    historical = any(
        cue in normalized for cue in HISTORICAL_CUES
    )

    # Check negation cues while excluding a few common
    # misleading constructions.
    negation_text = normalized

    for exception in [
        "not only",
        "not uncommon",
        "no change in",
    ]:
        negation_text = negation_text.replace(exception, " ")

    negated = any(
        cue in negation_text for cue in NEGATION_CUES
    )

    # Assign contextual category with conservative precedence.
    if family_history:
        status = "FAMILY_HISTORY"
    elif uncertain:
        status = "UNCERTAIN"
    elif negated:
        status = "NEGATED"
    elif historical:
        status = "HISTORICAL"
    else:
        status = "PRESENT_MENTION"

    return {
        "negated": bool(negated),
        "uncertain": bool(uncertain),
        "historical": bool(historical),
        "family_history": bool(family_history),
        "status": status,
    }


def attach_context(text: str, entities: list) -> list:
    """
    Associate each entity with its containing sentence
    and assign contextual attributes.
    """
    sentences = split_sentences(text)
    contextual_entities = []

    for entity in entities:
        sentence_info = next(
            (
                s for s in sentences
                if s["start"] <= entity["start"] < s["end"]
            ),
            None,
        )

        if sentence_info:
            sentence = sentence_info["text"]
        else:
            sentence = entity["text"]

        context = detect_clinical_context(sentence)

        contextual_entities.append({
            **entity,
            **context,
            "evidence_sentence": sentence,
        })

    return contextual_entities


# ============================================================
# MODULE 4: INFORMATION EXTRACTION
# ============================================================

def extract_cancer_information(text: str) -> dict:
    """
    Combine NER and contextual analysis into structured output.
    """
    clean_text = preprocess_clinical_text(text)

    entities = clinical_ner(clean_text)
    entities = attach_context(clean_text, entities)

    cancer_mentions = [
        e for e in entities
        if e["type"] in {"CANCER", "HISTOLOGY"}
    ]

    # This is a mention summary, not a patient diagnosis.
    present = [
        e for e in cancer_mentions
        if e["status"] == "PRESENT_MENTION"
    ]

    uncertain = [
        e for e in cancer_mentions
        if e["status"] == "UNCERTAIN"
    ]

    historical = [
        e for e in cancer_mentions
        if e["status"] == "HISTORICAL"
    ]

    negated = [
        e for e in cancer_mentions
        if e["status"] == "NEGATED"
    ]

    family = [
        e for e in cancer_mentions
        if e["status"] == "FAMILY_HISTORY"
    ]

    if present:
        report_status = "CANCER_MENTION_PRESENT"
    elif uncertain:
        report_status = "CANCER_MENTION_UNCERTAIN"
    elif historical:
        report_status = "HISTORICAL_CANCER_MENTION"
    elif negated:
        report_status = "CANCER_MENTION_NEGATED"
    elif family:
        report_status = "FAMILY_HISTORY_ONLY"
    else:
        report_status = "NO_MATCHED_CANCER_MENTION"

    return {
        "report_status": report_status,
        "entities": entities,
        "summary": {
            "present_mentions": len(present),
            "uncertain_mentions": len(uncertain),
            "historical_mentions": len(historical),
            "negated_mentions": len(negated),
            "family_history_mentions": len(family),
        },
        "note": (
            "A missing entity match does not establish that "
            "the patient does not have cancer."
        ),
    }


# ============================================================
# MODULE 5: OPTIONAL CANCER-TYPE CLASSIFIER
# ============================================================

def train_cancer_classifier(
    reports: pd.Series,
    labels: pd.Series,
) -> Pipeline:
    """
    Train a basic supervised classifier.

    labels must be verified cancer-type reference labels.
    Use patient-level train/test splitting for proper evaluation.
    """
    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_features=100_000,
                strip_accents="unicode",
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ])

    clean_reports = reports.apply(preprocess_clinical_text)
    model.fit(clean_reports, labels.astype(str))

    return model


def classify_cancer_type(
    text: str,
    model: Pipeline,
) -> dict:
    """Return predicted training label and model class scores."""
    clean_text = preprocess_clinical_text(text)

    probabilities = model.predict_proba([clean_text])[0]
    best_idx = probabilities.argmax()

    return {
        "predicted_class": str(model.classes_[best_idx]),
        "model_score": float(probabilities[best_idx]),
        "class_scores": {
            str(label): float(score)
            for label, score in zip(model.classes_, probabilities)
        },
        "warning": (
            "Model scores are not necessarily calibrated clinical "
            "probabilities and do not establish a diagnosis."
        ),
    }


# ============================================================
# MODULE 6: CLINICAL INTERPRETATION
# ============================================================

def interpret_report(extraction: dict) -> dict:
    """
    Create a cautious review summary based on extracted evidence.
    """
    status = extraction["report_status"]

    interpretations = {
        "CANCER_MENTION_PRESENT": (
            "Cancer-related terminology was detected without a "
            "recognized negation or uncertainty cue. Verify the "
            "source report and pathology conclusion."
        ),
        "CANCER_MENTION_UNCERTAIN": (
            "Cancer-related terminology occurs in an uncertain "
            "context. Do not treat this as confirmed cancer."
        ),
        "HISTORICAL_CANCER_MENTION": (
            "Historical cancer-related terminology was detected. "
            "Current disease status is not established."
        ),
        "CANCER_MENTION_NEGATED": (
            "Cancer-related terminology was detected in a sentence "
            "with a negation cue. Verify the scope and conclusion."
        ),
        "FAMILY_HISTORY_ONLY": (
            "Family-history context was detected. This is not "
            "evidence that the patient has cancer."
        ),
        "NO_MATCHED_CANCER_MENTION": (
            "No cancer-specific dictionary match was detected. "
            "This does not establish absence of cancer."
        ),
    }

    return {
        "report_status": status,
        "interpretation": interpretations[status],
        "requires_human_review": True,
        "diagnostic_conclusion": None,
    }


# ============================================================
# MODULE 7: FULL PIPELINE
# ============================================================

def run_clinical_nlp(text: str) -> dict:
    """Run preprocessing, NER, context, extraction and interpretation."""
    clean_text = preprocess_clinical_text(text)

    extraction = extract_cancer_information(clean_text)
    interpretation = interpret_report(extraction)

    return {
        "clean_text": clean_text,
        "extraction": extraction,
        "interpretation": interpretation,
    }


# ============================================================
# MODULE 8: TEST WITH EXAMPLE REPORTS
# ============================================================

if __name__ == "__main__":

    examples = [
        "Biopsy confirms adenocarcinoma of the colon.",
        "No evidence of malignancy in the sampled tissue.",
        "Possible metastatic carcinoma in the lung.",
        "History of breast cancer, previously treated.",
        "Family history of colon cancer.",
    ]

    for text in examples:
        result = run_clinical_nlp(text)

        print("\n" + "=" * 70)
        print("REPORT:", text)
        print(json.dumps(result, indent=2))



REPORT: Biopsy confirms adenocarcinoma of the colon.
{
  "clean_text": "Biopsy confirms adenocarcinoma of the colon.",
  "extraction": {
    "report_status": "CANCER_MENTION_PRESENT",
    "entities": [
      {
        "text": "adenocarcinoma",
        "type": "CANCER",
        "start": 16,
        "end": 30,
        "negated": false,
        "uncertain": false,
        "historical": false,
        "family_history": false,
        "status": "PRESENT_MENTION",
        "evidence_sentence": "Biopsy confirms adenocarcinoma of the colon."
      },
      {
        "text": "adenocarcinoma",
        "type": "HISTOLOGY",
        "start": 16,
        "end": 30,
        "negated": false,
        "uncertain": false,
        "historical": false,
        "family_history": false,
        "status": "PRESENT_MENTION",
        "evidence_sentence": "Biopsy confirms adenocarcinoma of the colon."
      },
      {
        "text": "colon",
        "type": "ANATOMICAL_SITE",
        "start": 38,
        "end"